# Dataset Creation for Mountain NER 

In [14]:
## install and import the necessary libraries to interact with the Google Gemini API.
# !pip install -q google-generativeai]
import json
import google.genai as genai
import os

In [ ]:
GOOGLE_API_KEY = "GOOGLE_API_KEY" #fake key
output_file = "himalayas_ner_dataset.json"

client = genai.Client(api_key=GOOGLE_API_KEY)

In [16]:
## choose promt type based on data exists

existing_data = []
if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        existing_data = json.load(f)
else:
    print("Output file haven't been created yet.")
    
num_existing = len(existing_data)

if num_existing == 0:
    prompt = """
        Act as a Data Engineer creating a dataset for an NLP Named Entity Recognition (NER) task. 
        Generate 50 UNIQUE English sentences describing expeditions, tourism, or geography. 

        Rules:
        1. Each sentence MUST contain exactly one name of a prominent mountain from anywhere in the world (e.g., Alps, Andes, Rockies, Caucasus, Himalayas).
        2. Crucially, mix the naming styles: sometimes use the "Mount X" format (e.g., Mount Blanc), and sometimes use the mountain name directly without any prefix (e.g., Everest, Hoverla, Kilimanjaro).
        3. Vary the position of the mountain name: place it at the beginning, in the middle, or at the end of sentences across different examples.
        4. In many sentences, include various surrounding geographical features (e.g., provinces, nearby straits, valleys, rivers, bays, or borders).
        5. Only the mountain name should be tagged as "B-MOUNTAIN" / "I-MOUNTAIN". ALL other geographical entities MUST be tagged as "O".
        6. Provide the output in a STRICT JSON array format.
        7. Do not include markdown formatting like ```json in the output, just the raw JSON array.
        8. Each object must have "tokens" (list of string words/punctuation) and "ner_tags" (list of BIO tags).
        
        Example of expected output structure:
        [
        {
            "tokens": ["Mount", "Fuji", "overlooks", "the", "Suruga", "Bay", "in", "Shizuoka", "Prefecture", "."],
            "ner_tags": ["B-MOUNTAIN", "I-MOUNTAIN", "O", "O", "O", "O", "O", "O", "O", "O"]
        }
        ]
    """ 

elif num_existing < 499:
    prompt = f"""
        Act as a Data Engineer creating a dataset for an NLP Named Entity Recognition (NER) task. 
        Generate 50 NEW and UNIQUE English sentences describing expeditions, tourism, or geography. 

        IMPORTANT: You have already generated {num_existing} sentences previously. 
        Do NOT repeat previous ideas. Use different contexts and a variety of mountain names.

        Rules:
        1. Each sentence MUST contain exactly one name of a prominent mountain from anywhere in the world (e.g., Alps, Andes, Rockies, Caucasus, Himalayas).
        2. Crucially, mix the naming styles: sometimes use the "Mount X" format (e.g., Mount Blanc), and sometimes use the mountain name directly without any prefix (e.g., Everest, Hoverla, Kilimanjaro).
        3. Vary the position of the mountain name: place it at the beginning, in the middle, or at the end of sentences across different examples.
        4. In many sentences, include various surrounding geographical features (e.g., provinces, nearby straits, valleys, rivers, bays, or borders).
        5. Only the mountain name should be tagged as "B-MOUNTAIN" / "I-MOUNTAIN". ALL other geographical entities MUST be tagged as "O".
        6. Provide the output in a STRICT JSON array format.
        7. Do not include markdown formatting like ```json in the output, just the raw JSON array.
        8. Each object must have "tokens" (list of string words/punctuation) and "ner_tags" (list of BIO tags).
    """

else:
    prompt = """
        Act as a Data Engineer creating a dataset for an NLP Named Entity Recognition (NER) task. 
        Generate 50 UNIQUE English sentences designed as "Hard Negatives" to prevent the model from overgeneralizing.
        
        Split the 50 sentences equally into FIVE categories (10 sentences each):
        
        CATEGORY 1: Non-geographical use of mountain names (brands, pet names, human names, companies).
        Example: "He named his dog Everest."
        
        CATEGORY 2: Other geographical entities (cities, countries, rivers, oceans, straits).
        Example: "The Bosphorus strait connects to the Black Sea."
        
        CATEGORY 3: Wordplay and common nouns. Use words like "mount", "mountain", "peak", or "summit" as regular verbs or metaphorical nouns.
        Example: "We need to mount the new TV on the wall." or "She reached the peak of her career."
        
        CATEGORY 4: Deceptive names. Use locations, universities, or media titles that contain the word "Mount" or "Mountain" but are NOT actual mountains.
        Example: "Google's office is in Mountain View." or "I watched Brokeback Mountain."
        
        CATEGORY 5: Other physical landforms. Use famous canyons, plateaus, hills, cliffs, or deserts.
        Example: "Tourists love the Grand Canyon and the Cliffs of Moher."
        
        Rules:
        1. VERY IMPORTANT: Because there are NO actual mountains acting as mountains in these sentences, their ner_tags MUST be completely "O". Do NOT use "B-MOUNTAIN" or "I-MOUNTAIN" anywhere.
        2. Provide the output in a STRICT JSON array format.
        3. Do not include markdown formatting like ```json in the output, just the raw JSON array.
        4. Each object must have "tokens" (list of string words/punctuation) and "ner_tags" (list of BIO tags).
    """

In [17]:
## generation of content via gemini-3.5-flash and writing in the file
try:
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt
    )
    raw_text = response.text.strip()

    if raw_text.startswith("```json"):
        raw_text = raw_text[7:]
    if raw_text.endswith("```"):
        raw_text = raw_text[:-3]

    new_data = json.loads(raw_text)
    print(f"{len(new_data)} sentences successfully generated")

    existing_data.extend(new_data)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(existing_data, f, indent=4, ensure_ascii=False)
        
except Exception as e:
    print(f"Error: {e}")

50 sentences successfully generated
